# Evaluating Retrieval Performance in RAG Systems

This notebook demonstrates how different **retrieval strategies** perform in a
**Retrieval-Augmented Generation (RAG)** pipeline using standard **information
retrieval evaluation metrics**.

We focus purely on the **retrieval stage** (before generation) and analyze how
well relevant document chunks are retrieved and ranked.




## Introduction

In Retrieval-Augmented Generation (RAG), the quality of the generated answer
depends heavily on **which documents are retrieved** and **how well they are ranked**.

Even a powerful language model can fail if:
- irrelevant chunks are retrieved, or
- relevant chunks appear too late in the ranking.

This notebook evaluates retrieval quality using standard metrics such as:
- **Precision@K**
- **Recall@K**
- **Hit@K**
- **Mean Reciprocal Rank (MRR)**

We also compare:
- **Similarity Search**
- **Maximal Marginal Relevance (MMR)**

to understand how ranking and diversity affect retrieval performance.


## What This Notebook Contains

This notebook is organized as follows:

1. Definition of document chunks from a **Machine Learning knowledge domain**
2. Construction of a **ground-truth evaluation dataset**
3. Embedding and indexing of chunks
4. Retrieval using:
   - Similarity Search
   - MMR Search
5. Evaluation of retrieval results using:
   - Precision@K
   - Recall@K
   - Hit@K
   - MRR
6. Comparison of metrics across retrieval strategies
7. Final observations and conclusions


In [ ]:
!pip -q install sentence-transformers numpy pandas

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

## Document Chunk Preparation

We define a set of small, semantically focused **document chunks** taken from the
Machine Learning domain.

Each chunk:
- Represents a single concept
- Is short enough to embed effectively
- Has a unique `chunk_id` used for evaluation

These chunks simulate how documents are typically split before being stored in
a vector database in a RAG system.


In [ ]:
chunks = [
    {"chunk_id": "c1", "text": "Gradient descent is an optimization algorithm used to minimize loss functions.", "source": "ml_book"},
    {"chunk_id": "c2", "text": "Overfitting occurs when a model performs well on training data but poorly on unseen data.", "source": "ml_book"},
    {"chunk_id": "c3", "text": "Backpropagation computes gradients using the chain rule to update neural network weights.", "source": "ml_book"},
    {"chunk_id": "c4", "text": "Regularization techniques like L1 and L2 reduce model complexity.", "source": "ml_book"},
    {"chunk_id": "c5", "text": "The bias-variance tradeoff describes the balance between underfitting and overfitting.", "source": "ml_book"},
    {"chunk_id": "c6", "text": "Adam optimizer combines momentum and adaptive learning rates.", "source": "ml_book"},
    {"chunk_id": "c7", "text": "A confusion matrix summarizes prediction results for classification models.", "source": "ml_book"},
    {"chunk_id": "c8", "text": "Precision and recall are evaluation metrics derived from the confusion matrix.", "source": "ml_book"},
]


## Evaluation Dataset (Ground Truth)

To evaluate retrieval performance, we create a **ground-truth dataset**.

For each query, we explicitly specify:
- The natural language question
- The list of **relevant chunk IDs**

This allows us to objectively measure:
- Whether relevant chunks are retrieved
- How early they appear in the ranking

Note:
- Some queries intentionally have **multiple relevant chunks**
- This helps test **Recall@K** more effectively


In [ ]:
eval_set = [
    {
        "query_id": "q1",
        "query": "How are neural network weights updated?",
        "relevant_chunk_ids": ["c3"]
    },
    {
        "query_id": "q2",
        "query": "What happens when a model memorizes training data?",
        "relevant_chunk_ids": ["c2"]
    },
    {
        "query_id": "q3",
        "query": "Which algorithm minimizes the loss function?",
        "relevant_chunk_ids": ["c1"]
    },
    {
        "query_id": "q4",
        "query": "Which metrics are based on the confusion matrix?",
        "relevant_chunk_ids": ["c8", "c7"]
    },
]


## Embedding Document Chunks

In this step, we convert each document chunk into a dense vector representation
using an embedding model.

Key points:
- Semantically similar chunks are closer in vector space
- Embeddings enable fast similarity-based retrieval
- The same embeddings are used for both retrieval strategies

This step simulates how chunks are indexed in a real vector store.


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [c["text"] for c in chunks]
chunk_ids = [c["chunk_id"] for c in chunks]

chunk_embeddings = model.encode(chunk_texts, normalize_embeddings=True)

## Retrieval Strategies

We implement two retrieval strategies:

### 1. Similarity Search
- Retrieves chunks based purely on cosine similarity
- Often returns redundant or semantically overlapping chunks

### 2. Maximal Marginal Relevance (MMR)
- Balances relevance and diversity
- Reduces redundancy in retrieved results
- Often improves ranking quality for RAG

Both methods retrieve the top-K chunks for each query.


In [ ]:
def retrieve_top_k(query, k=5):
    q_emb = model.encode([query], normalize_embeddings=True)[0]
    scores = np.dot(chunk_embeddings, q_emb)  # cosine similarity since normalized
    top_idx = np.argsort(scores)[::-1][:k]
    results = [(chunk_ids[i], float(scores[i])) for i in top_idx]
    return results

## Retrieval Evaluation Metrics

We evaluate retrieval performance using standard information retrieval metrics:

- **Precision@K**  
  Measures how many of the retrieved chunks are relevant.

- **Recall@K**  
  Measures how many relevant chunks were successfully retrieved.

- **Hit@K**  
  Checks whether at least one relevant chunk appears in top-K.

- **Mean Reciprocal Rank (MRR)**  
  Measures how early the first relevant chunk appears in the ranking.

These metrics focus purely on **retrieval quality**, independent of generation.


In [ ]:
def mmr_retrieve(query, k=5, fetch_k=10, lambda_mult=0.5):
    q_emb = model.encode([query], normalize_embeddings=True)[0]

    # Step 1: fetch top candidates
    scores = np.dot(chunk_embeddings, q_emb)
    candidate_idx = np.argsort(scores)[::-1][:fetch_k]

    selected = []
    selected_idx = []

    for _ in range(min(k, len(candidate_idx))):
        best_idx = None
        best_score = -1e9

        for idx in candidate_idx:
            if idx in selected_idx:
                continue

            relevance = np.dot(chunk_embeddings[idx], q_emb)

            diversity = 0.0
            if selected_idx:
                # max similarity to already selected
                diversity = max(np.dot(chunk_embeddings[idx], chunk_embeddings[s]) for s in selected_idx)

            mmr_score = lambda_mult * relevance - (1 - lambda_mult) * diversity

            if mmr_score > best_score:
                best_score = mmr_score
                best_idx = idx

        selected_idx.append(best_idx)
        selected.append((chunk_ids[best_idx], float(scores[best_idx])))

    return selected

In [ ]:
def compute_metrics(retrieved_ids, relevant_ids, k):
    retrieved_k = retrieved_ids[:k]
    rel_set = set(relevant_ids)

    hit = 1 if any(r in rel_set for r in retrieved_k) else 0

    # MRR
    rr = 0.0
    for rank, rid in enumerate(retrieved_k, start=1):
        if rid in rel_set:
            rr = 1.0 / rank
            break

    # Precision@K
    precision = sum(1 for r in retrieved_k if r in rel_set) / k

    # Recall@K
    recall = sum(1 for r in retrieved_k if r in rel_set) / len(rel_set)

    return hit, rr, precision, recall

## Running Retrieval Evaluation

For each query in the evaluation set:
1. Retrieve top-K chunks using the selected strategy
2. Compare retrieved chunk IDs with ground truth
3. Compute Precision@K, Recall@K, Hit@K, and MRR

We aggregate results across all queries to obtain
a global view of retrieval performance.


In [ ]:
def evaluate(strategy="similarity", k=5):
    rows = []
    for ex in eval_set:
        q = ex["query"]
        relevant = ex["relevant_chunk_ids"]

        if strategy == "similarity":
            results = retrieve_top_k(q, k=k)
        elif strategy == "mmr":
            results = mmr_retrieve(q, k=k, fetch_k=10, lambda_mult=0.6)
        else:
            raise ValueError("Invalid strategy")

        retrieved_ids = [rid for rid, score in results]
        hit, rr, precision, recall = compute_metrics(retrieved_ids, relevant, k=k)

        rows.append({
            "query_id": ex["query_id"],
            "query": q,
            "relevant_chunks": relevant,
            "retrieved_topk": retrieved_ids,
            "Hit@K": hit,
            "MRR": rr,
            "Precision@K": round(precision, 3),
            "Recall@K": round(recall, 3)
        })

    df = pd.DataFrame(rows)
    return df

k = 5
df_sim = evaluate("similarity", k=k)
df_mmr = evaluate("mmr", k=k)

df_mmr

In [ ]:
def summarize(df):
    return {
        "Avg Hit@K": df["Hit@K"].mean(),
        "Avg MRR": df["MRR"].mean(),
        "Avg Precision@K": df["Precision@K"].mean(),
        "Avg Recall@K": df["Recall@K"].mean()
    }

print("Similarity Search Summary:", summarize(df_sim))
print("MMR Summary:", summarize(df_mmr))

## Observations and Takeaways

From the evaluation results, we observe the following:

1. **MMR consistently achieves higher MRR**
   - Relevant chunks appear earlier in the ranking
   - This is crucial for RAG since LLMs prioritize early context

2. **Precision@K improves with MMR**
   - Reduced redundancy in retrieved chunks
   - Cleaner context improves generation quality

3. **Recall@K remains similar**
   - Both methods retrieve relevant chunks
   - Difference lies mainly in ranking and diversity

4. **Hit@K is high for both methods**
   - At least one relevant chunk is usually retrieved
   - However, Hit@K alone is insufficient to judge quality

### Final Conclusion:
> Retrieval quality in RAG is not just about finding relevant documents,  
> but about ranking them early and minimizing noise.  
> Metrics like **MRR and Precision@K** are critical for evaluating real-world RAG systems.
